In [522]:
from pathlib import Path

import numpy as np
import pandas as pd
from joblib import dump
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from tqdm.auto import tqdm

workspace_root = None
for candidate in [Path.cwd(), Path.cwd().parent]:
    if (candidate / "data" / "meteorology_dataset.csv").exists() and (candidate / "level2").exists():
        workspace_root = candidate
        break

if workspace_root is None:
    workspace_root = Path.cwd()

saved_models_dir = workspace_root / "level2" / "saved_models"
saved_models_dir.mkdir(parents=True, exist_ok=True)

def save_model_artifact(model_name, model, imputer, feature_columns, metadata=None):
    artifact_path = saved_models_dir / f"{model_name}.joblib"
    artifact = {
        "model": model,
        "imputer": imputer,
        "feature_columns": list(feature_columns),
        "excluded_features": list(excluded_features),
        "metadata": metadata or {},
    }
    dump(artifact, artifact_path)
    print(f"Saved {model_name} to {artifact_path}")
    return artifact_path

def location_group_key(frame):
    if "location" not in frame.columns:
        return pd.Series("__all__", index=frame.index, dtype="string")
    return frame["location"].fillna("__missing__").astype("string")

def grouped_shift(grouped, column_name, periods):
    if column_name not in grouped.obj.columns:
        return None
    return grouped[column_name].shift(periods)

def grouped_rolling(series, group_key, window, reducer):
    return (
        series.groupby(group_key)
        .rolling(window)
        .agg(reducer)
        .reset_index(level=0, drop=True)
    )

df = pd.read_csv("../data/meteorology_dataset.csv")
df_name = "meteorology_dataset.csv"
data = df.copy()

# Normalize headers once to avoid hidden-space mismatches.
data.columns = [str(col).strip() for col in data.columns]

time_col = "time"
temp_col = "temperature_2m"
dew_col = "dew_point_2m"
humidity_col = "relative_humidity_2m"

# User-controlled exclusions from the final training feature set.
excluded_features = ["dayofweek", "day", "wind_speed_10m", "time", "year"]

if time_col in data.columns:
    data[time_col] = pd.to_datetime(data[time_col], errors="coerce")
    data = data.sort_values(time_col).reset_index(drop=True)

if temp_col not in data.columns:
    raise KeyError(
        f"Could not find '{temp_col}'. Available columns: {list(data.columns)}"
    )

raw_location_count = int(data["location"].dropna().astype(str).nunique()) if "location" in data.columns else 1
location_key = location_group_key(data)

# Convert all non-time and non-location columns to numeric so all weather variables can be used.
for col in data.columns:
    if col != time_col and col != "location":
        data[col] = pd.to_numeric(data[col], errors="coerce")

grouped = data.groupby(location_key, sort=False, group_keys=False)

# Start from all dataset columns except target source column, timestamp, and location.
base_drop_cols = [c for c in [temp_col, time_col, "location"] if c in data.columns]
feature_df = data.drop(columns=base_drop_cols).copy()

# Remove columns that became entirely NaN after numeric conversion.
feature_df = feature_df.dropna(axis=1, how="all")

temp_shift_1 = grouped_shift(grouped, temp_col, 1)
dew_shift_1 = grouped_shift(grouped, dew_col, 1)
humidity_shift_1 = grouped_shift(grouped, humidity_col, 1)

# Keep engineered temperature history features within each location only.
for h in [1, 2, 3, 6, 12]:
    feature_df[f"temp_lag_{h}"] = grouped_shift(grouped, temp_col, h)
    if dew_col in data.columns:
        feature_df[f"dew_lag_{h}"] = grouped_shift(grouped, dew_col, h)

for h in [24, 48, 72, 96]:
    feature_df[f"temp_lag_{h}"] = grouped_shift(grouped, temp_col, h)
    if dew_col in data.columns:
        feature_df[f"dew_lag_{h}"] = grouped_shift(grouped, dew_col, h)
    feature_df[f"temp_roll_mean_{h}"] = grouped_rolling(temp_shift_1, location_key, h, "mean")

if time_col in data.columns:
    feature_df["hour"] = data[time_col].dt.hour
    feature_df["dayofweek"] = data[time_col].dt.dayofweek
    feature_df["month"] = data[time_col].dt.month

if dew_col in data.columns:
    feature_df["dew_lag_3"] = grouped_shift(grouped, dew_col, 3)
if humidity_col in data.columns:
    feature_df["humidity_lag_3"] = grouped_shift(grouped, humidity_col, 3)
    feature_df["humidity_roll_mean_6"] = grouped_rolling(humidity_shift_1, location_key, 6, "mean")
    feature_df["humidity_change"] = data[humidity_col] - humidity_shift_1
feature_df["monthly_temp_mean"] = grouped_rolling(temp_shift_1, location_key, 720, "mean")
feature_df["pressure_gap"] = data["pressure_msl"] - data["surface_pressure"]

# Apply exclusion list to final model features.
selected_feature_cols = [col for col in feature_df.columns if col not in excluded_features]
print(f"Selected features for modeling: {selected_feature_cols}")
feature_df = feature_df[selected_feature_cols].copy()

target = grouped_shift(grouped, temp_col, -1)
model_data = feature_df.copy()
model_data["target"] = target
trainable = model_data.dropna().copy()
X = trainable.drop(columns="target")
y = trainable["target"]

if time_col not in data.columns:
    raise ValueError("Walk-forward validation requires a valid time column.")

# Build day-based walk-forward folds over shared timestamps after location-safe feature engineering.
trainable_times = data.loc[trainable.index, time_col].dt.normalize()
unique_days = pd.Index(trainable_times.dropna().unique()).sort_values()

train_days = 16
val_days = 2
test_days = 2
window_days = train_days + val_days + test_days

if len(unique_days) < window_days:
    raise ValueError(
        f"Need at least {window_days} unique days for walk-forward validation, got {len(unique_days)}."
    )

walk_forward_splits = []
for start_idx in range(len(unique_days) - window_days + 1):
    train_day_block = unique_days[start_idx:start_idx + train_days]
    val_day_block = unique_days[start_idx + train_days:start_idx + train_days + val_days]
    test_day_block = unique_days[start_idx + train_days + val_days:start_idx + window_days]

    train_mask = trainable_times.isin(train_day_block)
    val_mask = trainable_times.isin(val_day_block)
    test_mask = trainable_times.isin(test_day_block)

    if train_mask.sum() == 0 or val_mask.sum() == 0 or test_mask.sum() == 0:
        continue

    walk_forward_splits.append({
        "train_days": train_day_block,
        "val_days": val_day_block,
        "test_days": test_day_block,
        "train_idx": X.index[train_mask],
        "val_idx": X.index[val_mask],
        "test_idx": X.index[test_mask],
    })

if not walk_forward_splits:
    raise ValueError("No valid walk-forward splits could be created from the dataset.")

# Keep the latest fold as the active split so downstream cells continue to work unchanged.
active_split = walk_forward_splits[-1]
X_train = X.loc[active_split["train_idx"]]
X_val = X.loc[active_split["val_idx"]]
X_test = X.loc[active_split["test_idx"]]
y_train = y.loc[active_split["train_idx"]]
y_val = y.loc[active_split["val_idx"]]
y_test = y.loc[active_split["test_idx"]]

# Optional combined train+validation set for later experiments.
X_trainval_full = pd.concat([X_train, X_val])
y_trainval_full = pd.concat([y_train, y_val])

imputer = SimpleImputer(strategy="median")
X_train_imputed = pd.DataFrame(
    imputer.fit_transform(X_train), index=X_train.index, columns=X_train.columns
)
X_val_imputed = pd.DataFrame(
    imputer.transform(X_val), index=X_val.index, columns=X_val.columns
)
X_test_imputed = pd.DataFrame(
    imputer.transform(X_test), index=X_test.index, columns=X_test.columns
)

next_step_features = feature_df.loc[[feature_df.index[-1]], X.columns].copy()
if not next_step_features.isna().all(axis=None):
    next_step_features_imputed = pd.DataFrame(
        imputer.transform(next_step_features),
        index=next_step_features.index,
        columns=next_step_features.columns,
    )
else:
    next_step_features_imputed = None

print(f"Walk-forward folds created: {len(walk_forward_splits)}")
print(f"Locations kept separate for lags/rolling: {raw_location_count}")
print(
    f"Active fold -> train: {active_split['train_days'][0].date()} to {active_split['train_days'][-1].date()}, "
    f"val: {active_split['val_days'][0].date()} to {active_split['val_days'][-1].date()}, "
    f"test: {active_split['test_days'][0].date()} to {active_split['test_days'][-1].date()}"
)
print(f"Rows -> train: {len(X_train)}, val: {len(X_val)}, test: {len(X_test)}")

Selected features for modeling: ['relative_humidity_2m', 'dew_point_2m', 'rain', 'cloud_cover', 'cloud_cover_low', 'cloud_cover_mid', 'cloud_cover_highh', 'wind_direction_10m', 'wind_gusts_10m', 'wind_direction_100m', 'wind_speed_100m', 'pressure_msl', 'surface_pressure', 'temp_lag_1', 'dew_lag_1', 'temp_lag_2', 'dew_lag_2', 'temp_lag_3', 'dew_lag_3', 'temp_lag_6', 'dew_lag_6', 'temp_lag_12', 'dew_lag_12', 'temp_lag_24', 'dew_lag_24', 'temp_roll_mean_24', 'temp_lag_48', 'dew_lag_48', 'temp_roll_mean_48', 'temp_lag_72', 'dew_lag_72', 'temp_roll_mean_72', 'temp_lag_96', 'dew_lag_96', 'temp_roll_mean_96', 'hour', 'month', 'humidity_lag_3', 'humidity_roll_mean_6', 'humidity_change', 'monthly_temp_mean', 'pressure_gap']
Walk-forward folds created: 317
Locations kept separate for lags/rolling: 18
Active fold -> train: 2026-02-14 to 2026-03-01, val: 2026-03-02 to 2026-03-03, test: 2026-03-04 to 2026-03-05
Rows -> train: 6912, val: 864, test: 846


## Linear Regression Model

In [523]:
class MyLinearRegression:
    def __init__(self):
        self.coef_ = None
        self.intercept_ = None

    def fit(self, X, y):
        X_np = np.asarray(X, dtype=float)
        y_np = np.asarray(y, dtype=float).reshape(-1, 1)

        # Add bias term
        X_b = np.c_[np.ones((X_np.shape[0], 1)), X_np]

        # Normal equation with pseudo-inverse for stability
        theta = np.linalg.pinv(X_b.T @ X_b) @ X_b.T @ y_np

        self.intercept_ = float(theta[0, 0])
        self.coef_ = theta[1:, 0]
        return self

    def predict(self, X):
        X_np = np.asarray(X, dtype=float)
        return self.intercept_ + X_np @ self.coef_

import sklearn.linear_model

linear_model = sklearn.linear_model.LinearRegression()
linear_model.fit(X_train_imputed, y_train)

y_pred_val_lr = linear_model.predict(X_val_imputed)
y_pred_test_lr = linear_model.predict(X_test_imputed)

mae_val_lr = mean_absolute_error(y_val, y_pred_val_lr)
rmse_val_lr = mean_squared_error(y_val, y_pred_val_lr) ** 0.5
r2_val_lr = r2_score(y_val, y_pred_val_lr)

mae_test_lr = mean_absolute_error(y_test, y_pred_test_lr)
rmse_test_lr = mean_squared_error(y_test, y_pred_test_lr) ** 0.5
r2_test_lr = r2_score(y_test, y_pred_test_lr)

print("Linear Regression results")
print(f"Validation rows: {len(X_val)}")
print(f"Validation MAE:  {mae_val_lr:.3f}")
print(f"Validation RMSE: {rmse_val_lr:.3f}")
print(f"Validation R²:   {r2_val_lr:.3f}")
print(f"Test rows: {len(X_test)}")
print(f"Test MAE:  {mae_test_lr:.3f}")
print(f"Test RMSE: {rmse_test_lr:.3f}")
print(f"Test R²:   {r2_test_lr:.3f}")

forecast_results_lr = X_test.copy()
forecast_results_lr["actual_temperature_next_step"] = y_test.values
forecast_results_lr["predicted_temperature_next_step_lr"] = y_pred_test_lr

if next_step_features_imputed is not None:
    next_temperature_forecast_lr = float(linear_model.predict(next_step_features_imputed)[0])
    print(f"Next-step temperature forecast (Linear Regression): {next_temperature_forecast_lr:.3f}")
else:
    next_temperature_forecast_lr = None
    print("Next-step forecast could not be computed because the last row has missing features.")

linear_model_artifact_path = save_model_artifact(
    model_name="linear_regression",
    model=linear_model,
    imputer=imputer,
    feature_columns=X_train.columns,
    metadata={
        "validation_metrics": {"mae": mae_val_lr, "rmse": rmse_val_lr, "r2": r2_val_lr},
        "test_metrics": {"mae": mae_test_lr, "rmse": rmse_test_lr, "r2": r2_test_lr},
        "next_temperature_forecast": next_temperature_forecast_lr,
    },
)

forecast_results_lr.head()

Linear Regression results
Validation rows: 864
Validation MAE:  0.541
Validation RMSE: 0.703
Validation R²:   0.960
Test rows: 846
Test MAE:  0.550
Test RMSE: 0.724
Test R²:   0.956
Next-step temperature forecast (Linear Regression): 6.212
Saved linear_regression to c:\Users\Pedro Passos\Desktop\Weather-Prediction\level2\saved_models\linear_regression.joblib


,relative_humidity_2m,dew_point_2m,rain,cloud_cover,cloud_cover_low,cloud_cover_mid,cloud_cover_highh,wind_direction_10m,wind_gusts_10m,wind_direction_100m,...,temp_roll_mean_96,hour,month,humidity_lag_3,humidity_roll_mean_6,humidity_change,monthly_temp_mean,pressure_gap,actual_temperature_next_step,predicted_temperature_next_step_lr
157248,76,8.9,0.0,0,0,0,0,9,26.6,18,...,12.912500,0,3,65.0,66.666667,3.0,13.394028,1.2,12.5,12.572933
157249,91,8.1,0.0,0,0,0,0,93,16.9,67,...,11.562500,0,3,81.0,80.500000,0.0,11.615972,5.0,9.2,9.198745
157250,70,5.8,0.0,1,1,0,0,28,28.8,50,...,9.510417,0,3,66.0,63.166667,6.0,8.916111,55.0,10.8,10.683508
157251,81,8.7,0.0,2,0,2,0,119,8.6,83,...,11.776042,0,3,80.0,76.500000,5.0,12.049722,1.1,10.8,11.304716
157252,75,7.5,0.0,6,0,0,5,25,28.1,36,...,11.426042,0,3,71.0,70.833333,0.0,11.645556,33.8,11.6,11.429066


## XGBoost Model

In [524]:
from xgboost import XGBRegressor

xgb_model = XGBRegressor(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(
    X_train_imputed, y_train,
    eval_set=[(X_val_imputed, y_val)],
    verbose=False
)

y_pred_val_xgb = xgb_model.predict(X_val_imputed)
y_pred_test_xgb = xgb_model.predict(X_test_imputed)

mae_val_xgb = mean_absolute_error(y_val, y_pred_val_xgb)
rmse_val_xgb = mean_squared_error(y_val, y_pred_val_xgb) ** 0.5
r2_val_xgb = r2_score(y_val, y_pred_val_xgb)

mae_test_xgb = mean_absolute_error(y_test, y_pred_test_xgb)
rmse_test_xgb = mean_squared_error(y_test, y_pred_test_xgb) ** 0.5
r2_test_xgb = r2_score(y_test, y_pred_test_xgb)

print("XGBoost Regressor results")
print(f"Validation rows: {len(X_val)}")
print(f"Validation MAE:  {mae_val_xgb:.3f}")
print(f"Validation RMSE: {rmse_val_xgb:.3f}")
print(f"Validation R²:   {r2_val_xgb:.3f}")
print(f"Test rows: {len(X_test)}")
print(f"Test MAE:  {mae_test_xgb:.3f}")
print(f"Test RMSE: {rmse_test_xgb:.3f}")
print(f"Test R²:   {r2_test_xgb:.3f}")

forecast_results_xgb = X_test.copy()
forecast_results_xgb["actual_temperature_next_step"] = y_test.values
forecast_results_xgb["predicted_temperature_next_step_xgb"] = y_pred_test_xgb

if next_step_features_imputed is not None:
    next_temperature_forecast_xgb = float(xgb_model.predict(next_step_features_imputed)[0])
    print(f"Next-step temperature forecast (XGBoost): {next_temperature_forecast_xgb:.3f}")
else:
    next_temperature_forecast_xgb = None
    print("Next-step forecast could not be computed because the last row has missing features.")

xgb_model_artifact_path = save_model_artifact(
    model_name="xgboost_regressor",
    model=xgb_model,
    imputer=imputer,
    feature_columns=X_train.columns,
    metadata={
        "validation_metrics": {"mae": mae_val_xgb, "rmse": rmse_val_xgb, "r2": r2_val_xgb},
        "test_metrics": {"mae": mae_test_xgb, "rmse": rmse_test_xgb, "r2": r2_test_xgb},
        "next_temperature_forecast": next_temperature_forecast_xgb,
    },
)

forecast_results_xgb.head()

XGBoost Regressor results
Validation rows: 864
Validation MAE:  0.564
Validation RMSE: 0.756
Validation R²:   0.954
Test rows: 846
Test MAE:  0.547
Test RMSE: 0.751
Test R²:   0.953
Next-step temperature forecast (XGBoost): 5.861
Saved xgboost_regressor to c:\Users\Pedro Passos\Desktop\Weather-Prediction\level2\saved_models\xgboost_regressor.joblib


,relative_humidity_2m,dew_point_2m,rain,cloud_cover,cloud_cover_low,cloud_cover_mid,cloud_cover_highh,wind_direction_10m,wind_gusts_10m,wind_direction_100m,...,temp_roll_mean_96,hour,month,humidity_lag_3,humidity_roll_mean_6,humidity_change,monthly_temp_mean,pressure_gap,actual_temperature_next_step,predicted_temperature_next_step_xgb
157248,76,8.9,0.0,0,0,0,0,9,26.6,18,...,12.912500,0,3,65.0,66.666667,3.0,13.394028,1.2,12.5,12.594431
157249,91,8.1,0.0,0,0,0,0,93,16.9,67,...,11.562500,0,3,81.0,80.500000,0.0,11.615972,5.0,9.2,9.020168
157250,70,5.8,0.0,1,1,0,0,28,28.8,50,...,9.510417,0,3,66.0,63.166667,6.0,8.916111,55.0,10.8,10.887836
157251,81,8.7,0.0,2,0,2,0,119,8.6,83,...,11.776042,0,3,80.0,76.500000,5.0,12.049722,1.1,10.8,11.373585
157252,75,7.5,0.0,6,0,0,5,25,28.1,36,...,11.426042,0,3,71.0,70.833333,0.0,11.645556,33.8,11.6,11.207242


## LightGBM Model

In [525]:
from lightgbm import LGBMRegressor

lgbm_model = LGBMRegressor(
    n_estimators=2000,
    learning_rate=0.02,
    num_leaves=63,
    min_child_samples=50,
    subsample=0.8,
    colsample_bytree=0.7,
    reg_alpha=0.1,
    reg_lambda=1.0,
    force_col_wise=True,
    verbosity=-1,
    random_state=42,
    n_jobs=-1
)

lgbm_model.fit(
    X_train_imputed,
    y_train,
    eval_set=[(X_val_imputed, y_val)],
    eval_metric="l2",
)

y_pred_val_lgbm = lgbm_model.predict(X_val_imputed)
y_pred_test_lgbm = lgbm_model.predict(X_test_imputed)

mae_val_lgbm = mean_absolute_error(y_val, y_pred_val_lgbm)
rmse_val_lgbm = mean_squared_error(y_val, y_pred_val_lgbm) ** 0.5
r2_val_lgbm = r2_score(y_val, y_pred_val_lgbm)

mae_test_lgbm = mean_absolute_error(y_test, y_pred_test_lgbm)
rmse_test_lgbm = mean_squared_error(y_test, y_pred_test_lgbm) ** 0.5
r2_test_lgbm = r2_score(y_test, y_pred_test_lgbm)

print("LightGBM Regressor results")
print(f"Validation rows: {len(X_val)}")
print(f"Validation MAE:  {mae_val_lgbm:.3f}")
print(f"Validation RMSE: {rmse_val_lgbm:.3f}")
print(f"Validation R²:   {r2_val_lgbm:.3f}")
print(f"Test rows: {len(X_test)}")
print(f"Test MAE:  {mae_test_lgbm:.3f}")
print(f"Test RMSE: {rmse_test_lgbm:.3f}")
print(f"Test R²:   {r2_test_lgbm:.3f}")

forecast_results_lgbm = X_test.copy()
forecast_results_lgbm["actual_temperature_next_step"] = y_test.values
forecast_results_lgbm["predicted_temperature_next_step_lgbm"] = y_pred_test_lgbm

if next_step_features_imputed is not None:
    next_temperature_forecast_lgbm = float(lgbm_model.predict(next_step_features_imputed)[0])
    print(f"Next-step temperature forecast (LightGBM): {next_temperature_forecast_lgbm:.3f}")
else:
    next_temperature_forecast_lgbm = None
    print("Next-step forecast could not be computed because the last row has missing features.")

lgbm_model_artifact_path = save_model_artifact(
    model_name="lightgbm_regressor",
    model=lgbm_model,
    imputer=imputer,
    feature_columns=X_train.columns,
    metadata={
        "validation_metrics": {"mae": mae_val_lgbm, "rmse": rmse_val_lgbm, "r2": r2_val_lgbm},
        "test_metrics": {"mae": mae_test_lgbm, "rmse": rmse_test_lgbm, "r2": r2_test_lgbm},
        "next_temperature_forecast": next_temperature_forecast_lgbm,
    },
)

forecast_results_lgbm.head()

LightGBM Regressor results
Validation rows: 864
Validation MAE:  0.546
Validation RMSE: 0.732
Validation R²:   0.956
Test rows: 846
Test MAE:  0.531
Test RMSE: 0.727
Test R²:   0.956
Next-step temperature forecast (LightGBM): 5.827
Saved lightgbm_regressor to c:\Users\Pedro Passos\Desktop\Weather-Prediction\level2\saved_models\lightgbm_regressor.joblib


,relative_humidity_2m,dew_point_2m,rain,cloud_cover,cloud_cover_low,cloud_cover_mid,cloud_cover_highh,wind_direction_10m,wind_gusts_10m,wind_direction_100m,...,temp_roll_mean_96,hour,month,humidity_lag_3,humidity_roll_mean_6,humidity_change,monthly_temp_mean,pressure_gap,actual_temperature_next_step,predicted_temperature_next_step_lgbm
157248,76,8.9,0.0,0,0,0,0,9,26.6,18,...,12.912500,0,3,65.0,66.666667,3.0,13.394028,1.2,12.5,12.757865
157249,91,8.1,0.0,0,0,0,0,93,16.9,67,...,11.562500,0,3,81.0,80.500000,0.0,11.615972,5.0,9.2,8.768535
157250,70,5.8,0.0,1,1,0,0,28,28.8,50,...,9.510417,0,3,66.0,63.166667,6.0,8.916111,55.0,10.8,10.819001
157251,81,8.7,0.0,2,0,2,0,119,8.6,83,...,11.776042,0,3,80.0,76.500000,5.0,12.049722,1.1,10.8,10.980156
157252,75,7.5,0.0,6,0,0,5,25,28.1,36,...,11.426042,0,3,71.0,70.833333,0.0,11.645556,33.8,11.6,11.110520


In [526]:
# Refit deployable artifacts on all clean rows after selecting model settings on the active walk-forward split.
X_full = X.copy()
y_full = y.copy()

imputer_full = SimpleImputer(strategy="median")
X_full_imputed = pd.DataFrame(
    imputer_full.fit_transform(X_full), index=X_full.index, columns=X_full.columns
)

next_step_features_full = feature_df.loc[[feature_df.index[-1]], X_full.columns].copy()
if not next_step_features_full.isna().all(axis=None):
    next_step_features_full_imputed = pd.DataFrame(
        imputer_full.transform(next_step_features_full),
        index=next_step_features_full.index,
        columns=next_step_features_full.columns,
    )
else:
    next_step_features_full_imputed = None

production_specs = [
    (
        "linear_regression",
        sklearn.linear_model.LinearRegression(),
        {
            "validation_metrics": {"mae": mae_val_lr, "rmse": rmse_val_lr, "r2": r2_val_lr},
            "test_metrics": {"mae": mae_test_lr, "rmse": rmse_test_lr, "r2": r2_test_lr},
        },
    ),
    (
        "xgboost_regressor",
        XGBRegressor(
            n_estimators=400,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            n_jobs=-1,
        ),
        {
            "validation_metrics": {"mae": mae_val_xgb, "rmse": rmse_val_xgb, "r2": r2_val_xgb},
            "test_metrics": {"mae": mae_test_xgb, "rmse": rmse_test_xgb, "r2": r2_test_xgb},
        },
    ),
    (
        "lightgbm_regressor",
        LGBMRegressor(
            n_estimators=2000,
            learning_rate=0.02,
            num_leaves=63,
            min_child_samples=50,
            subsample=0.8,
            colsample_bytree=0.7,
            reg_alpha=0.1,
            reg_lambda=1.0,
            force_col_wise=True,
            verbosity=-1,
            random_state=42,
            n_jobs=-1,
        ),
        {
            "validation_metrics": {"mae": mae_val_lgbm, "rmse": rmse_val_lgbm, "r2": r2_val_lgbm},
            "test_metrics": {"mae": mae_test_lgbm, "rmse": rmse_test_lgbm, "r2": r2_test_lgbm},
        },
    ),
]

production_summary = []
for model_name, model, metrics in production_specs:
    model.fit(X_full_imputed, y_full)
    next_temp = None
    if next_step_features_full_imputed is not None:
        next_temp = float(model.predict(next_step_features_full_imputed)[0])

    artifact_path = save_model_artifact(
        model_name=model_name,
        model=model,
        imputer=imputer_full,
        feature_columns=X_full.columns,
        metadata={
            **metrics,
            "training_rows": int(len(X_full)),
            "retrained_on_full_data": True,
            "next_temperature_forecast": next_temp,
        },
    )
    production_summary.append({
        "model": model_name,
        "training_rows": int(len(X_full)),
        "artifact_path": str(artifact_path),
        "next_temperature_forecast": next_temp,
    })

production_summary = pd.DataFrame(production_summary)
print("Production artifacts refreshed on full clean dataset.")
production_summary

Saved linear_regression to c:\Users\Pedro Passos\Desktop\Weather-Prediction\level2\saved_models\linear_regression.joblib
Saved xgboost_regressor to c:\Users\Pedro Passos\Desktop\Weather-Prediction\level2\saved_models\xgboost_regressor.joblib
Saved lightgbm_regressor to c:\Users\Pedro Passos\Desktop\Weather-Prediction\level2\saved_models\lightgbm_regressor.joblib
Production artifacts refreshed on full clean dataset.


,model,training_rows,artifact_path,next_temperature_forecast
0,linear_regression,145134,c:\Users\Pedro Passos\Desktop\Weather-Predicti...,6.070855
1,xgboost_regressor,145134,c:\Users\Pedro Passos\Desktop\Weather-Predicti...,5.863490
2,lightgbm_regressor,145134,c:\Users\Pedro Passos\Desktop\Weather-Predicti...,5.659137


## Mixed Voting Model (Best one)

In [527]:
# Average the test predictions from Linear Regression, XGBoost, and LightGBM.
# Use index alignment so the ensemble only scores rows available in all three model outputs.

from joblib import load as load_artifact

lr_test = forecast_results_lr[
    ["actual_temperature_next_step", "predicted_temperature_next_step_lr"]
].rename(columns={"predicted_temperature_next_step_lr": "pred_lr"})

xgb_test = forecast_results_xgb[
    ["actual_temperature_next_step", "predicted_temperature_next_step_xgb"]
].rename(columns={"predicted_temperature_next_step_xgb": "pred_xgb"})

lgbm_test = forecast_results_lgbm[
    ["actual_temperature_next_step", "predicted_temperature_next_step_lgbm"]
].rename(columns={"predicted_temperature_next_step_lgbm": "pred_lgbm"})

forecast_results_vote = (
    lr_test[["actual_temperature_next_step", "pred_lr"]]
    .join(xgb_test[["pred_xgb"]], how="inner")
    .join(lgbm_test[["pred_lgbm"]], how="inner")
    .copy()
 )

forecast_results_vote["predicted_temperature_next_step_vote"] = forecast_results_vote[
    ["pred_lr", "pred_xgb", "pred_lgbm"]
] .mean(axis=1)

mae_test_vote = mean_absolute_error(
    forecast_results_vote["actual_temperature_next_step"],
    forecast_results_vote["predicted_temperature_next_step_vote"]
)
rmse_test_vote = mean_squared_error(
    forecast_results_vote["actual_temperature_next_step"],
    forecast_results_vote["predicted_temperature_next_step_vote"]
) ** 0.5
r2_test_vote = r2_score(
    forecast_results_vote["actual_temperature_next_step"],
    forecast_results_vote["predicted_temperature_next_step_vote"]
)

print("Voting/Averaging Ensemble results")
print(f"Test rows used: {len(forecast_results_vote)}")
print(f"Test MAE:  {mae_test_vote:.3f}")
print(f"Test RMSE: {rmse_test_vote:.3f}")
print(f"Test R²:   {r2_test_vote:.3f}")

next_temperature_forecast_vote = np.mean([
    next_temperature_forecast_lr,
    next_temperature_forecast_xgb,
    next_temperature_forecast_lgbm
])
print(f"Next-step temperature forecast (Voting/Average): {next_temperature_forecast_vote:.3f}")

linear_artifact = load_artifact(linear_model_artifact_path)
xgb_artifact = load_artifact(xgb_model_artifact_path)
lgbm_artifact = load_artifact(lgbm_model_artifact_path)

ensemble_feature_columns = lgbm_artifact.get("featureColumns") or lgbm_artifact.get("feature_columns")
ensemble_imputer = lgbm_artifact.get("imputer")
ensemble_models = [
    ("linear_regression", linear_artifact["model"]),
    ("xgboost_regressor", xgb_artifact["model"]),
    ("lightgbm_regressor", lgbm_artifact["model"]),
]

mixed_model_artifact_path = saved_models_dir / "voting_average_ensemble.joblib"
mixed_model_artifact = {
    "modelBundle": {
        "type": "vote",
        "family": "saved",
        "models": ensemble_models,
    },
    "imputer": ensemble_imputer,
    "feature_columns": list(ensemble_feature_columns),
    "excluded_features": list(excluded_features),
    "metadata": {
        "validation_metrics": {"mae": mae_test_vote, "rmse": rmse_test_vote, "r2": r2_test_vote},
        "test_metrics": {"mae": mae_test_vote, "rmse": rmse_test_vote, "r2": r2_test_vote},
        "next_temperature_forecast": next_temperature_forecast_vote,
    },
}
dump(mixed_model_artifact, mixed_model_artifact_path)
print(f"Saved voting_average_ensemble to {mixed_model_artifact_path}")

forecast_results_vote.head()

Voting/Averaging Ensemble results
Test rows used: 846
Test MAE:  0.469
Test RMSE: 0.647
Test R²:   0.965
Next-step temperature forecast (Voting/Average): 5.967
Saved voting_average_ensemble to c:\Users\Pedro Passos\Desktop\Weather-Prediction\level2\saved_models\voting_average_ensemble.joblib


,actual_temperature_next_step,pred_lr,pred_xgb,pred_lgbm,predicted_temperature_next_step_vote
157248,12.5,12.572933,12.594431,12.757865,12.641743
157249,9.2,9.198745,9.020168,8.768535,8.995816
157250,10.8,10.683508,10.887836,10.819001,10.796782
157251,10.8,11.304716,11.373585,10.980156,11.219485
157252,11.6,11.429066,11.207242,11.110520,11.248943
